# 🔬 Ablation: Có vs. Không Có `ContextGateValidator` (`32B-AWQ` Outline + Coordinator + LLM Validator | `7B` Chưa Finetune Viết Truyện)

**Kiến trúc chạy an toàn (Process-Isolated Subprocess — tránh triệt để lỗi CUDA fork & rò rỉ VRAM trên Colab):**
- **Outline (`OutlineAgent`):** `Qwen/Qwen2.5-32B-Instruct-AWQ` (`float16`, `gpu_mem=0.85`)
- **Planning & Critic (`Condition D`, không Freytag):** Python CPU (`0` LLM calls)
- **Coordinator (`CoordinatorAgent`):** `Qwen/Qwen2.5-32B-Instruct-AWQ`
- **Validator (`ContextGateValidator`):** 3 nhánh so song song trên cùng 3 Macro Plans MoPS:
  - `no_validator`: Bỏ hẳn Validator
  - `with_validator_heuristic`: Python Heuristic (`threshold=0.80`)
  - `with_validator_llm`: **`Qwen/Qwen2.5-32B-Instruct-AWQ`** (`temp=0.0`, `threshold=0.80`)
- **Writer (chưa finetune):** `Qwen/Qwen2.5-7B-Instruct`
- **Mỗi lượt đổi model (`32B ↔ 7B`) chạy trong 1 subprocess Python sạch độc lập** (`VLLM_WORKER_MULTIPROC_METHOD=spawn`), tự giải phóng 100% VRAM và lưu checkpoint sau từng chương (`--resume`) vào Google Drive!

---

## Bước 1: Kiểm tra GPU, Mount Google Drive & Clone `MA-GRPO`

In [ ]:
!nvidia-smi

import os
import sys
import subprocess
import getpass
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation")
except Exception:
    OUTPUT_DIR = Path("ablation_context_gate_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục lưu kết quả trên Drive: {OUTPUT_DIR}")

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"

if os.path.exists("/content"):
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    except Exception:
        token = None

    if not os.path.exists(REPO_DIR):
        if token:
            subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
        else:
            res = subprocess.run(["git", "clone", f"https://{REPO_URL}", REPO_DIR])
            if res.returncode != 0:
                token = getpass.getpass("Nhập GitHub Personal Access Token: ")
                subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

assert os.path.exists("scripts/run_context_gate_ablation.py"), "Không tìm thấy scripts/run_context_gate_ablation.py!"
print("Working dir:", os.getcwd())
!git log -n 1 --oneline


## Bước 2: Cài đặt `vllm` & Thư viện cần thiết

In [ ]:
!pip install -q vllm pandas pyyaml pydantic
print("✅ Cài đặt hoàn tất!")


## Bước 3: Chạy Trọn Gói Ablation Qua Subprocess Độc Lập (Auto-Resume, 0 OOM, Streaming Log)
- **Không gọi `torch.cuda` / `vllm` trực tiếp trong cell Colab** — mỗi pha chạy trong tiến trình Python sạch riêng biệt nên tự động xả 100% VRAM và không bao giờ dính lỗi fork CUDA.
- Nếu rớt mạng giữa chừng, chạy lại cell này sẽ **tự nối tiếp đúng chương đang dở (`--resume`)**.

In [ ]:
import os
import subprocess
import sys

PLANNER_COORD_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"
WRITER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
NUM_STORIES = 3
NUM_CHAPTERS = 8
RUN_ARMS = ["no_validator", "with_validator_heuristic", "with_validator_llm"]

cmd = [
    sys.executable, "-u", "scripts/run_context_gate_ablation.py",
    "--phase", "all",
    "--backend", "vllm",
    "--planner-model", PLANNER_COORD_MODEL,
    "--writer-model", WRITER_MODEL,
    "--planner-gpu-mem", "0.85",
    "--writer-gpu-mem", "0.80",
    "--coord-max-len", "12288",
    "--writer-max-len", "16384",
    "--premises-file", "data/mops_premises.jsonl",
    "--exclude-test-file", "data/tell-me-a-story-test.jsonl",
    "--num-stories", str(NUM_STORIES),
    "--num-chapters", str(NUM_CHAPTERS),
    "--target-words-choices", "8000", "9000", "10000",
    "--arms", *RUN_ARMS,
    "--validator-threshold", "0.80",
    "--output-dir", str(OUTPUT_DIR),
    "--resume",
]

env = os.environ.copy()
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
env["VLLM_LOGGING_LEVEL"] = "WARNING"

print("Running command:\n", " ".join(cmd), "\n", flush=True)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"run_context_gate_ablation.py exited with code {ret}")
print("\n✅ Hoàn tất sinh toàn bộ truyện 7B cho mọi nhánh!")


## Bước 4: Xem Bảng Tóm Tắt & Danh Sách File Đã Lưu Trên Google Drive

In [ ]:
import json
import pandas as pd

summary_file = OUTPUT_DIR / "ablation_summary.json"
rows = json.loads(summary_file.read_text(encoding="utf-8"))
print("=== TÓM TẮT KẾT QUẢ SINH TRUYỆN 7B (ĐÃ LƯU GOOGLE DRIVE) ===")
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== CÁC FILE ĐÃ LƯU TRÊN GOOGLE DRIVE ===")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  - {p} ({p.stat().st_size / 1024:.1f} KB)")


## Bước 5: Tự động Ngắt Server Colab Sau Khi Lưu Xong Lên Drive

In [ ]:
import time

AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("✅ Toàn bộ truyện 7B đã lưu vào Google Drive! Chờ 15 giây đồng bộ rồi tự ngắt runtime...")
    time.sleep(15)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Không thể gọi runtime.unassign(): {e}")
else:
    print("ℹ️ AUTO_DISCONNECT=False — Nhớ tắt runtime thủ công sau khi kiểm tra.")
